# Steel Defect Detection in Sugarcane Mill Machinery

## Modules

In [ ]:
import os
import csv
import random
import numpy as np
import pandas as pd
import cv2

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import networkx as nx

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.nn.functional as F   
import torch.optim as optim

In [ ]:
import timm

## Configs

### Random Seed

In [ ]:
random_seed = 42
torch.manual_seed(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

### CUDA Check

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CUDA Available:", torch.cuda.is_available())
print("Device:", device)

### Dataset Directory

In [ ]:
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), "..\\\\.."))
DATA_DIR = os.path.join(ROOT_DIR, "Datasets", "steel-defect-detection")
TRAIN_CSV_PATH = os.path.join(DATA_DIR, "train.csv")
TRAIN_IMAGES_DIR = os.path.join(DATA_DIR, "train_images")
TEST_IMAGES_DIR = os.path.join(DATA_DIR, "test_images")

print(f"""
CSV Path: {(TRAIN_CSV_PATH, os.path.exists(TRAIN_CSV_PATH))}
Train Images Directory: {(TRAIN_IMAGES_DIR, os.path.exists(TRAIN_IMAGES_DIR))}
Test Images Directory: {(TEST_IMAGES_DIR, os.path.exists(TEST_IMAGES_DIR))}
"""
)


### Image Configs

In [ ]:
NUM_CLASSES = 4
IMG_HEIGHT = 256
IMG_WIDTH = 1600
BATCH_SIZE = 8
IMAGE_SIZE = (IMG_HEIGHT, IMG_HEIGHT)

### Training Configs

In [ ]:
THRESHOLD = 0.5
EPOCHS = 10
MAX_EPOCHS = 50
PATIENCE = 7
MODEL_NAME = "swinv2_base_window8_256"

### Saving the Model

In [ ]:
BEST_MODEL_DIRECTORY = os.path.join(ROOT_DIR, "Models", "Steel", MODEL_NAME)
print(f"Best Model Directory: {BEST_MODEL_DIRECTORY}")
# Create directory if it doesn't exist
os.makedirs(BEST_MODEL_DIRECTORY, exist_ok=True)
print("Directory created (if it didn't exist):", BEST_MODEL_DIRECTORY)

In [ ]:
## Fetch latest name of best model from BEST_MODEL_DIRECTORY and increment the number in the name by 1, to create new BEST_MODEL_PATH
existing_models = [f for f in os.listdir(BEST_MODEL_DIRECTORY) if f.endswith(".pth")]
if existing_models:
    existing_models.sort()
    latest_model = existing_models[-1]
    latest_number = int(latest_model.split(".")[0])
    new_number = latest_number + 1
else:
    new_number = 1

In [ ]:
BEST_MODEL_PATH = str(new_number) + ".pth"
BEST_MODEL_PATH

In [ ]:
BEST_MODEL_PATH = os.path.join(BEST_MODEL_DIRECTORY, BEST_MODEL_PATH)
BEST_MODEL_PATH = os.path.abspath(BEST_MODEL_PATH)
print("Best model path:", BEST_MODEL_PATH)

### Results Directory

In [ ]:
RESULTS_DIR = os.path.join(ROOT_DIR, "Results")
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS_FILE = os.path.join(RESULTS_DIR, "model_results_steel.csv")
print("Results file path:", RESULTS_FILE)

## Data Preprocessing

In [ ]:
df = pd.read_csv(TRAIN_CSV_PATH)

In [ ]:
df.head(10)

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
no_of_images = df["ImageId"].nunique()
print(f"Number of unique images: {no_of_images}")


### Image and Defect Class Relationship

Expected to have 4 entries for each image corresponding to each class of defect (1, 2, 3, 4), regardless of whether or not the particular image has a defect of said class ID.

In [ ]:
df['ImageId'].value_counts().head(10)

In [ ]:
print(f"Count of Images with not all categories of class defect data available: {int((df['ImageId'].value_counts() != 4).sum())}/{no_of_images}")

### Creating Full Dataset

In [ ]:
all_images = df['ImageId'].unique()
all_classes = [1, 2, 3, 4]

In [ ]:
full_index = pd.MultiIndex.from_product(
    [all_images, all_classes],
    names=['ImageId', 'ClassId']
)

In [ ]:
df_full = (
    df.set_index(['ImageId', 'ClassId'])
      .reindex(full_index)
      .reset_index()
)

In [ ]:
df_full.shape

In [ ]:
df_full.head(10)

In [ ]:
df_full['ImageId'].value_counts().head()

In [ ]:
print(f"Count of images with not all categories of class defect data available: {int((df_full['ImageId'].value_counts() != 4).sum())}/{no_of_images}")

In [ ]:
print(f"Defect presence in percentage: {df_full['EncodedPixels'].isna().mean() * 100:.2f}%")

In [ ]:
OUTPUT_CSV_PATH = os.path.join(DATA_DIR, "train_full.csv")
df_full.to_csv(OUTPUT_CSV_PATH, index=False)

### Essentials

In [ ]:
image_level_defects = (
    df_full.groupby('ImageId')['EncodedPixels']
           .apply(lambda x: x.notna().any())
)
image_level_defects.value_counts()

In [ ]:
df_full['has_defect'] = ~df_full['EncodedPixels'].isna()

class_distribution = (
    df_full.groupby('ClassId')['has_defect']
      .mean()
      .rename('defect_ratio')
)

print("Per-Class Defect Frequency", class_distribution, sep="\n")

In [ ]:
print(f"No. of unique images:", df_full['ImageId'].nunique())
print(f"No. of images with no defects:", image_level_defects.value_counts().sum())

In [ ]:
train_images_dir = os.path.join(DATA_DIR, "train_images")

missing_files = [
    img for img in df['ImageId'].unique()
    if not os.path.exists(os.path.join(train_images_dir, img))
]

print("Missing image files count:", len(missing_files))

### Run Length Decoder

Run Length Encoding (RLE) is a simple lossless compression technique that represents consecutive occurrences of the same value as a single value–count pair, for example encoding a long sequence of identical pixels as “(value, run length)” instead of storing each pixel individually. 

It is used here because the mage-processing pipeline contains large contiguous regions with uniform or near-uniform values, such as background areas, binary masks, segmentation maps, or thresholded images, and RLE exploits this spatial redundancy to reduce storage, transmission bandwidth, and memory footprint without losing any information.

The encoded data has to be decoded appropriately into a binary mask via the run length decoder below after the output from the vision module is obtained. For this purpose, run length decoding is performed.

In [ ]:
def rle_decode(rle_string, height=IMG_HEIGHT, width=IMG_WIDTH):
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle_string):
        return mask.reshape((width, height)).T

    rle = list(map(int, rle_string.split()))
    starts = rle[0::2]
    lengths = rle[1::2]

    for start, length in zip(starts, lengths):
        start -= 1
        mask[start:start + length] = 1

    return mask.reshape((width, height)).T


### Channel Mask Construction

Builds a 4-channel mask (4, H, W) for a single image, each channel for a defect class.

In [ ]:
def build_mask_for_image(image_id, df_full):

    mask = np.zeros((NUM_CLASSES, IMG_HEIGHT, IMG_WIDTH), dtype=np.uint8)

    image_rows = df_full[df_full['ImageId'] == image_id]

    for _, row in image_rows.iterrows():
        class_idx = int(row['ClassId']) - 1
        mask[class_idx] = rle_decode(
            row['EncodedPixels'],
            IMG_HEIGHT,
            IMG_WIDTH
        )

    return mask


### Sanity Check

In [ ]:
sample_image_id = df_full['ImageId'].iloc[0]

image_path = os.path.join(TRAIN_IMAGES_DIR, sample_image_id)
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

mask = build_mask_for_image(sample_image_id, df_full)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis("off")

for i in range(4):
    axes[i + 1].imshow(mask[i], cmap="gray")
    axes[i + 1].set_title(f"Defect Class {i + 1}")
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()


## Knowledge Graph

In [ ]:
G = nx.DiGraph()

### Nodes

#### Input Nodes

In [ ]:
layer_input = [
    "Defect_Class_1_Detected",
    "Defect_Class_2_Detected",
    "Defect_Class_3_Detected",
    "Defect_Class_4_Detected"
]

#### Attribute Nodes

In [ ]:
layer_attributes = [
    "Defect_Area",
    "Defect_Length",
    "Defect_Density",
    "Defect_Distribution"
]

#### Defect Interpretation Nodes

In [ ]:
layer_interpretation = [
    "Isolated_Minor_Defect",
    "Localized_Severe_Defect",
    "Widespread_Defect_Pattern",
    "Critical_Structural_Defect"
]

#### Quality Assessment Nodes

In [ ]:
layer_quality = [
    "Acceptable_Quality",
    "Marginal_Quality",
    "Unacceptable_Quality"
]

#### Decision and Action Nodes

In [ ]:
layer_decision = [
    "Accept_Strip",
    "Downgrade_Strip",
    "Reject_Strip",
    "Manual_Inspection_Required"
]

### Edges

#### Input to Attribute Edges

In [ ]:
edges_evidence_to_attr = [
    ("Defect_Class_1_Detected", "Defect_Area", "if present"),
    ("Defect_Class_2_Detected", "Defect_Area", "if present"),
    ("Defect_Class_3_Detected", "Defect_Density", "if clustered"),
    ("Defect_Class_4_Detected", "Defect_Length", "if elongated"),

    ("Defect_Class_1_Detected", "Defect_Distribution", ""),
    ("Defect_Class_2_Detected", "Defect_Distribution", ""),
    ("Defect_Class_3_Detected", "Defect_Distribution", ""),
    ("Defect_Class_4_Detected", "Defect_Distribution", "")
]

#### Attribute to Interpretation Edges

In [ ]:
edges_attr_to_interp = [
    ("Defect_Area", "Isolated_Minor_Defect", "area < T1"),
    ("Defect_Area", "Localized_Severe_Defect", "area ≥ T2"),
    ("Defect_Length", "Critical_Structural_Defect", "length ≥ T3"),
    ("Defect_Density", "Widespread_Defect_Pattern", "density ≥ T4"),
    ("Defect_Distribution", "Widespread_Defect_Pattern", "distributed")
]

#### Interpretation to Quality Edges

In [ ]:
edges_interp_to_quality = [
    ("Isolated_Minor_Defect", "Acceptable_Quality", ""),
    ("Localized_Severe_Defect", "Marginal_Quality", ""),
    ("Widespread_Defect_Pattern", "Unacceptable_Quality", ""),
    ("Critical_Structural_Defect", "Unacceptable_Quality", "")
]

#### Quality to Decision Edges

In [ ]:
edges_quality_to_decision = [
    ("Acceptable_Quality", "Accept_Strip", ""),
    ("Marginal_Quality", "Downgrade_Strip", ""),
    ("Unacceptable_Quality", "Reject_Strip", "")
]

### Adding Nodes and Edges to Graph

In [ ]:
for node in layer_input:
    G.add_node(node, layer=1)
    
for node in layer_attributes:
    G.add_node(node, layer=2)
    
for node in layer_interpretation:
    G.add_node(node, layer=3)

for node in layer_quality:
    G.add_node(node, layer=4)

for node in layer_decision:
    G.add_node(node, layer=5)

for src, dst, cond in edges_evidence_to_attr:
    G.add_edge(src, dst, condition=cond)

for src, dst, cond in edges_attr_to_interp:
    G.add_edge(src, dst, condition=cond)

for src, dst, cond in edges_interp_to_quality:
    G.add_edge(src, dst, condition=cond)

for src, dst, cond in edges_quality_to_decision:
    G.add_edge(src, dst, condition=cond)

G.add_edge(
    "Critical_Structural_Defect",
    "Manual_Inspection_Required",
    condition="safety-critical"
)

### Visualization

In [ ]:
def layered_layout(G):
    pos = {}
    layer_nodes = {}

    for node, data in G.nodes(data=True):
        layer = data["layer"]
        layer_nodes.setdefault(layer, []).append(node)

    for layer, nodes in layer_nodes.items():
        for i, node in enumerate(nodes):
            pos[node] = (i, -layer)

    return pos

In [ ]:
layer_colors = {
    1: "#AED6F1",  # Input Evidence (light blue)
    2: "#A9DFBF",  # Spatial / Severity Attributes (light green)
    3: "#F9E79F",  # Interpretation (light yellow)
    4: "#F5CBA7",  # Quality Assessment (light orange)
    5: "#F1948A"   # Decision & Action (light red)
}

In [ ]:
nodes_by_layer = {}

for node, data in G.nodes(data=True):
    layer = data["layer"]
    nodes_by_layer.setdefault(layer, []).append(node)

In [ ]:
pos = layered_layout(G)

plt.figure(figsize=(20, 14))

for layer, nodes in nodes_by_layer.items():
    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=nodes,
        node_color=layer_colors[layer],
        node_size=3000,
        edgecolors="black",
        linewidths=1.0,
        label=f"Layer {layer}"
    )

nx.draw_networkx_edges(
    G,
    pos,
    arrows=True,
    arrowsize=15
)

nx.draw_networkx_labels(
    G,
    pos,
    font_size=9,
    font_weight="bold"
)

edge_labels = {
    (u, v): d["condition"]
    for u, v, d in G.edges(data=True)
    if d["condition"]
}

nx.draw_networkx_edge_labels(
    G,
    pos,
    edge_labels=edge_labels,
    font_size=8
)

legend_patches = [
    Patch(facecolor=layer_colors[1], edgecolor="black", label="Input Evidence"),
    Patch(facecolor=layer_colors[2], edgecolor="black", label="Spatial & Severity Attributes"),
    Patch(facecolor=layer_colors[3], edgecolor="black", label="Defect Interpretation"),
    Patch(facecolor=layer_colors[4], edgecolor="black", label="Quality Assessment"),
    Patch(facecolor=layer_colors[5], edgecolor="black", label="Decision & Action"),
]

plt.legend(
    handles=legend_patches,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08), 
    ncol=3,
    frameon=True,
    fontsize=9
)

plt.title("Steel Defect Detection — Logical Knowledge Graph", fontsize=14)
plt.axis("off")

## Save the graph visualization to Media (as Steel_Defect_Detection_KG.png)
output_path = os.path.join(ROOT_DIR, "Media", "Steel_Defect_Detection_KG.png")
plt.savefig(output_path)

plt.show()


## Dataset

### Dataset Class

In [ ]:
class SteelSegmentationDataset(Dataset):
    def __init__(self, df_full, image_dir, image_size):
        self.df_full = df_full
        self.image_dir = image_dir
        self.image_ids = df_full['ImageId'].unique()
        self.image_size = image_size

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]

        # Load image
        img_path = os.path.join(self.image_dir, image_id)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Resize image
        image = cv2.resize(image, (self.image_size[1], self.image_size[0]))

        image = image.astype(np.float32) / 255.0
        image = torch.from_numpy(image).permute(2, 0, 1)

        # Build mask
        mask = np.zeros((4, 256, 1600), dtype=np.uint8)

        rows = self.df_full[self.df_full['ImageId'] == image_id]

        for _, row in rows.iterrows():
            class_idx = int(row['ClassId']) - 1
            decoded = rle_decode(row['EncodedPixels'])
            mask[class_idx] = decoded

        # Resize mask (nearest to preserve binary)
        resized_mask = np.zeros((4, self.image_size[0], self.image_size[1]))

        for c in range(4):
            resized_mask[c] = cv2.resize(
                mask[c],
                (self.image_size[1], self.image_size[0]),
                interpolation=cv2.INTER_NEAREST
            )

        mask_tensor = torch.from_numpy(resized_mask).float()

        return image, mask_tensor

### Train Dataset

In [ ]:
full_train_dataset = SteelSegmentationDataset(
    df_full=df_full,
    image_dir=TRAIN_IMAGES_DIR,
    image_size=IMAGE_SIZE
)

print("No. of train images:", len(full_train_dataset))

### Train-Validation Split

In [ ]:
train_size = int(0.7 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(random_seed)
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

### Dataloaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

### After Batch Division

In [ ]:
images, masks = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Mask batch shape:", masks.shape)

## Model

### swinv2_base_window8_256

In [ ]:
class SegFormerLike(nn.Module):

    def __init__(self, num_classes=4):
        super().__init__()

        self.backbone = timm.create_model(
            MODEL_NAME,
            pretrained=True,
            features_only=True
        )

        in_channels = self.backbone.feature_info[-1]["num_chs"]
        self.conv = nn.Conv2d(in_channels, num_classes, kernel_size=1)

    def forward(self, x):

        features = self.backbone(x)[-1]

        if features.dim() == 4 and features.shape[-1] != features.shape[1]:
            features = features.permute(0, 3, 1, 2)

        x = F.interpolate(features, size=x.shape[-2:], mode="bilinear")

        return self.conv(x)

In [ ]:
model = SegFormerLike(num_classes=NUM_CLASSES)
print(model)

### Move to GPU

In [ ]:
model = model.to(device)
print("Model moved to:", device)

### Trainable Parameters

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Training

### Loss Function

Using only Binary Cross Entropy often leads to poor segmentation of thin defects.

Therefore we combine: BCE + Dice

BCE: Handles pixel-wise classification.

Dice Loss: Measures overlap between predicted and true masks.

#### Dice Loss

In [ ]:
# class DiceLoss(nn.Module):
    
#     def __init__(self, smooth=1):
#         super().__init__()
#         self.smooth = smooth

#     def forward(self, preds, targets):

#         preds = torch.sigmoid(preds)

#         preds = preds.view(-1)
#         targets = targets.view(-1)

#         intersection = (preds * targets).sum()

#         dice = (2. * intersection + self.smooth) / \
#                (preds.sum() + targets.sum() + self.smooth)

#         return 1 - dice

class DiceLoss(nn.Module):

    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):

        preds = torch.sigmoid(preds)

        # flatten per batch, per class
        preds = preds.reshape(preds.size(0), preds.size(1), -1)
        targets = targets.reshape(targets.size(0), targets.size(1), -1)

        intersection = (preds * targets).sum(dim=2)
        union = preds.sum(dim=2) + targets.sum(dim=2)

        dice = (2 * intersection + self.smooth) / (union + self.smooth)

        # average over classes and batch
        return 1 - dice.mean()

#### BCE-Dice

In [ ]:
# class BCEDiceLoss(nn.Module):

#     def __init__(self):
#         super().__init__()
#         self.bce = nn.BCEWithLogitsLoss()
#         self.dice = DiceLoss()

#     def forward(self, preds, targets):

#         bce_loss = self.bce(preds, targets)
#         dice_loss = self.dice(preds, targets)

#         return bce_loss + dice_loss

class BCEDiceLoss(nn.Module):

    def __init__(self, pos_weight=None):
        super().__init__()

        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.dice = DiceLoss()

    def forward(self, preds, targets):

        bce_loss = self.bce(preds, targets)
        dice_loss = self.dice(preds, targets)

        return bce_loss + dice_loss

#### Dice Score

In [ ]:
# def dice_score(preds, targets, threshold=0.5):

#     preds = torch.sigmoid(preds)
#     preds = (preds > threshold).float()

#     intersection = (preds * targets).sum()
#     union = preds.sum() + targets.sum()

#     return (2 * intersection) / (union + 1e-8)

def dice_score(preds, targets, threshold=THRESHOLD):

    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()

    preds = preds.reshape(preds.size(0), preds.size(1), -1)
    targets = targets.reshape(targets.size(0), targets.size(1), -1)

    intersection = (preds * targets).sum(dim=2)
    union = preds.sum(dim=2) + targets.sum(dim=2)

    dice = (2 * intersection) / (union + 1e-8)

    dice[union == 0] = 1.0

    return dice.mean()


def dice_per_class(preds, targets, threshold=THRESHOLD):

    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()

    # reshape: (B, C, H, W) → (B, C, N)
    preds = preds.reshape(preds.size(0), preds.size(1), -1)
    targets = targets.reshape(targets.size(0), targets.size(1), -1)

    intersection = (preds * targets).sum(dim=2)
    union = preds.sum(dim=2) + targets.sum(dim=2)

    dice = (2 * intersection) / (union + 1e-8)

    dice[union == 0] = 1.0

    # shape: (B, C)
    return dice

#### IoU Score

In [ ]:
def iou_score(preds, targets, threshold=0.5):

    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()

    intersection = (preds * targets).sum()
    union = preds.sum() + targets.sum() - intersection

    return intersection / (union + 1e-8)

In [ ]:
# criterion = BCEDiceLoss()

pos_weight = torch.tensor([3.0, 3.0, 2.0, 1.0]).to(device).view(1, 4, 1, 1)

criterion = BCEDiceLoss(pos_weight=pos_weight)

### Optimizer

In [ ]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

### Train Function

In [ ]:
def train_epoch(model, loader, optimizer, criterion):

    model.train()

    running_loss = 0

    for images, masks in loader:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, masks)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

### Validate Function

In [ ]:
# def validate_epoch(model, loader, criterion):

#     model.eval()

#     running_loss = 0
#     dice_total = 0
#     iou_total = 0

#     with torch.no_grad():

#         for images, masks in loader:

#             images = images.to(device)
#             masks = masks.to(device)

#             outputs = model(images)

#             loss = criterion(outputs, masks)

#             running_loss += loss.item()

#             dice_total += dice_score(outputs, masks)
#             iou_total += iou_score(outputs, masks)

#     avg_loss = running_loss / len(loader)
#     avg_dice = dice_total / len(loader)
#     avg_iou = iou_total / len(loader)

#     return avg_loss, avg_dice, avg_iou

def validate_epoch(model, loader, criterion):

    model.eval()

    running_loss = 0
    dice_total = 0
    iou_total = 0

    # NEW: per-class accumulator
    per_class_dice_total = torch.zeros(4, device=device)

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = criterion(outputs, masks)
            running_loss += loss.item()

            # overall Dice
            batch_dice = dice_score(outputs, masks)
            dice_total += batch_dice

            # IoU
            iou_total += iou_score(outputs, masks)

            # NEW: per-class Dice
            dice_pc = dice_per_class(outputs, masks)   # (B, C)
            per_class_dice_total += dice_pc.mean(dim=0)  # (C,)

    avg_loss = running_loss / len(loader)
    avg_dice = dice_total / len(loader)
    avg_iou = iou_total / len(loader)

    avg_per_class_dice = per_class_dice_total / len(loader)

    return avg_loss, avg_dice, avg_iou, avg_per_class_dice

### Training Loop

In [ ]:
best_val_dice = 0.0
best_val_dice_pc = torch.zeros(4)

epochs_without_improvement = 0

train_losses = []
val_losses = []
val_dices = []
val_dices_per_class = []

# print(
#     f"{'Epoch':<8}"
#     f"{'Train Loss':<15}"
#     f"{'Val Loss':<15}"
#     f"{'Dice':<12}"
#     f"{'IoU':<12}"
#     f"{'Status':<20}"
# )

# print("-" * 80)

print(
    f"{'Epoch':<6}"
    f"{'Train Loss':<14}"
    f"{'Val Loss':<14}"
    f"{'Dice':<10}"
    f"{'IoU':<10}"
    f"{'C1':<8}{'C2':<8}{'C3':<8}{'C4':<8}"
    f"{'Status':<20}"
)

print("-" * 110)

for epoch in range(1, MAX_EPOCHS + 1):

    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    # val_loss, val_dice, val_iou = validate_epoch(
    #     model,
    #     val_loader,
    #     criterion
    # )

    val_loss, val_dice, val_iou, val_dice_pc = validate_epoch(
        model,
        val_loader,
        criterion
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # val_dices.append(val_dice)
    val_dices.append(val_dice.item())
    val_dices_per_class.append(val_dice_pc.cpu().numpy())

    if val_dice > best_val_dice:

        best_val_dice = val_dice
        epochs_without_improvement = 0

        torch.save(model.state_dict(), BEST_MODEL_PATH)

        best_val_dice_pc = val_dice_pc.clone().detach()

        status = "New Best Model"

    else:

        epochs_without_improvement += 1
        status = f"No Improvement ({epochs_without_improvement}/{PATIENCE})"

    # print(
    #     f"{epoch:<8}"
    #     f"{train_loss:<15.4f}"
    #     f"{val_loss:<15.4f}"
    #     f"{val_dice:<12.4f}"
    #     f"{val_iou:<12.4f}"
    #     f"{status:<20}"
    # )

    c1, c2, c3, c4 = val_dice_pc.tolist()

    print(
        f"{epoch:<6}"
        f"{train_loss:<14.4f}"
        f"{val_loss:<14.4f}"
        f"{val_dice:<10.4f}"
        f"{val_iou:<10.4f}"
        f"{c1:<8.4f}{c2:<8.4f}{c3:<8.4f}{c4:<8.4f}"
        f"{status:<20}"
    )

    if epochs_without_improvement >= PATIENCE:

        print("\nEarly stopping triggered.")
        print(f"Best validation Dice: {best_val_dice:.4f}")
        break

## Results

### Load Best Model

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()

print("Best Validation Dice =", best_val_dice)

### Add Results to CSV

In [ ]:
# epochs_trained = len(train_losses)

# row = [
#     MODEL_NAME,
#     round(best_val_dice.item(), 5),
#     epochs_trained
# ]

# file_exists = os.path.isfile(RESULTS_FILE)

# with open(RESULTS_FILE, mode="a", newline="") as f:
#     writer = csv.writer(f)

#     # Write header only once
#     if not file_exists:
#         writer.writerow([
#             "model_name",
#             "best_val_dice",
#             "epochs_trained"
#         ])

#     writer.writerow(row)

# print(f"Results logged successfully to {RESULTS_FILE}.")
# print("Logged row:", row)

epochs_trained = len(train_losses)

# Convert per-class dice to list
c1, c2, c3, c4 = best_val_dice_pc.cpu().numpy()

row = [
    MODEL_NAME,
    round(best_val_dice.item(), 5),
    round(float(c1), 5),
    round(float(c2), 5),
    round(float(c3), 5),
    round(float(c4), 5),
    epochs_trained
]

file_exists = os.path.isfile(RESULTS_FILE)

with open(RESULTS_FILE, mode="a", newline="") as f:
    writer = csv.writer(f)

    # Write header only once
    if not file_exists:
        writer.writerow([
            "model_name",
            "best_val_dice",
            "dice_c1",
            "dice_c2",
            "dice_c3",
            "dice_c4",
            "epochs_trained"
        ])

    writer.writerow(row)

print(f"Results logged successfully to {RESULTS_FILE}.")
print("Logged row:", row)

### Loss and Dice Score Curves

In [ ]:
epochs_range = range(1, len(train_losses) + 1)

plt.figure(figsize=(10,4))

# Loss curves
plt.subplot(1,2,1)
plt.plot(epochs_range, train_losses, label="Train Loss")
plt.plot(epochs_range, val_losses, label="Val Loss")
plt.legend()
plt.title("Loss Curve")

# Dice curves
plt.subplot(1,2,2)
plt.plot(epochs_range, val_dices, label="Val Dice")
plt.legend()
plt.title("Dice Score Curve")

best_epoch = val_dices.index(max(val_dices)) + 1
plt.axvline(best_epoch, color='r', linestyle='--', label='Best Epoch')

plt.tight_layout()
plt.show()